# 03 — File I/O & Serialization


## 1. Basic File Operations

In [ ]:
import tempfile, os
from pathlib import Path

# Create a temp file for demos
tmp = Path(tempfile.mktemp(suffix='.txt'))
tmp.write_text('line1\nline2\nline3\n', encoding='utf-8')

# read() — entire file
with open(tmp) as f:
    print('read():', repr(f.read()))

# readline()
with open(tmp) as f:
    print('readline():', repr(f.readline()))
    print('readline():', repr(f.readline()))

# readlines()
with open(tmp) as f:
    print('readlines():', f.readlines())

# Iterate (most memory-efficient)
print('iterate:')
with open(tmp) as f:
    for line in f:
        print(' ', repr(line))

## 2. pathlib

In [ ]:
from pathlib import Path

p = Path('my-python') / 'README.md'
print(f'name:    {p.name}')
print(f'stem:    {p.stem}')
print(f'suffix:  {p.suffix}')
print(f'parent:  {p.parent}')
print(f'exists:  {p.exists()}')
print(f'is_file: {p.is_file()}')

# List all .md files
md_files = list(Path('my-python').glob('**/*.md'))
print(f'\nFound {len(md_files)} .md files')
for f in md_files[:5]:
    print(f'  {f}')

## 3. JSON

In [ ]:
import json
from datetime import datetime

data = {
    'name': 'Alice',
    'scores': [95, 87, 92],
    'active': True,
    'address': None,
    'meta': {'created': '2024-01-01'}
}

# Serialize
s = json.dumps(data, indent=2, sort_keys=True)
print(s)

# Deserialize
obj = json.loads(s)
print(type(obj), obj['name'])

In [ ]:
# Custom encoder for non-serializable types
import json
from datetime import date, datetime
from decimal import Decimal

class ExtendedEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (date, datetime)):
            return obj.isoformat()
        if isinstance(obj, Decimal):
            return float(obj)
        if isinstance(obj, set):
            return sorted(obj)
        return super().default(obj)

data = {
    'date': datetime.now(),
    'price': Decimal('9.99'),
    'tags': {'python', 'coding'}
}
print(json.dumps(data, cls=ExtendedEncoder, indent=2))

## 4. Pickle

In [ ]:
import pickle
import tempfile

# Pickle can serialize any Python object
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __repr__(self):
        return f'Point({self.x}, {self.y})'

data = {'point': Point(1, 2), 'func': len, 'nums': list(range(5))}

# Serialize to bytes
blob = pickle.dumps(data, protocol=pickle.HIGHEST_PROTOCOL)
print(f'Pickled size: {len(blob)} bytes')

# Deserialize
loaded = pickle.loads(blob)
print(loaded)
print(loaded['point'])
print(loaded['func']([1,2,3]))  # len([1,2,3]) = 3

## 5. CSV

In [ ]:
import csv, io

# Write to string buffer
buf = io.StringIO()
writer = csv.DictWriter(buf, fieldnames=['name', 'score', 'grade'])
writer.writeheader()
writer.writerows([
    {'name': 'Alice', 'score': 95, 'grade': 'A'},
    {'name': 'Bob',   'score': 87, 'grade': 'B'},
    {'name': 'Carol', 'score': 92, 'grade': 'A'},
])

csv_content = buf.getvalue()
print(csv_content)

# Read back
buf.seek(0)
reader = csv.DictReader(buf)
for row in reader:
    print(f"{row['name']:10} {row['score']:>5} {row['grade']}")

## 6. Performance: Reading Strategies

In [ ]:
import tempfile, time
from pathlib import Path

# Create a large test file
tmp = Path(tempfile.mktemp(suffix='.txt'))
tmp.write_text('\n'.join(f'line {i}' for i in range(100_000)))

# Strategy 1: read() — loads all into memory
start = time.perf_counter()
with open(tmp) as f:
    lines = f.read().splitlines()
t1 = time.perf_counter() - start

# Strategy 2: readlines()
start = time.perf_counter()
with open(tmp) as f:
    lines = f.readlines()
t2 = time.perf_counter() - start

# Strategy 3: iterate (lazy)
start = time.perf_counter()
count = 0
with open(tmp) as f:
    for line in f:
        count += 1
t3 = time.perf_counter() - start

print(f'read().splitlines(): {t1*1000:.2f}ms')
print(f'readlines():         {t2*1000:.2f}ms')
print(f'iterate (lazy):      {t3*1000:.2f}ms')
tmp.unlink()

## 7. Atomic Write Pattern

In [ ]:
import os, tempfile
from pathlib import Path

def atomic_write(path: str, content: str, encoding: str = 'utf-8') -> None:
    """Write file atomically — no partial writes visible to readers."""
    dir_ = Path(path).parent
    dir_.mkdir(parents=True, exist_ok=True)
    fd, tmp_path = tempfile.mkstemp(dir=dir_)
    try:
        with os.fdopen(fd, 'w', encoding=encoding) as f:
            f.write(content)
        os.replace(tmp_path, path)  # atomic rename
    except Exception:
        os.unlink(tmp_path)
        raise

atomic_write('/tmp/test_atomic.txt', 'Hello, atomic world!')
print(Path('/tmp/test_atomic.txt').read_text())